In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.geometry import AirfoilGeometry
from src.physics import AeroSolver
import numpy as np
import optuna
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# Fixed parameters
CHORD = 1 
ALPHA = 0
N_POINTS_DATASET = 100
N_TRIALS_PER_OPTI = 100
dataset = []

for i in range(N_POINTS_DATASET):
    v_target = np.random.uniform(50, 250)    # Velocity between 10 et 45 m/s
    alt_target = np.random.uniform(0, 12000) # Altitude between 0 et 2500 m
    study = optuna.create_study(direction="maximize")

    def objective(trial):
        m = trial.suggest_int("m", 0, 9)
        p = trial.suggest_int("p", 0, 9)
        t = trial.suggest_int("t", 1, 40)
        
        af = AirfoilGeometry(n_points=100)
        xu, yu, xl, yl = af.generate_naca4(m_int=m, p_int=p, t_int=t)
        x_coords, y_coords = af.get_coords_for_solver(xu, yu, xl, yl)
        
        solver = AeroSolver(altitude=alt_target, velocity=v_target, chord=CHORD)
        res = solver.solve(x_coords, y_coords, alpha=ALPHA)
        
        cl = res['cl'][0]
        cd = res['cd'][0]
        
        if cd <= 0: return 0
        
        return cl / cd

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study.optimize(objective, n_trials=N_TRIALS_PER_OPTI, show_progress_bar=False)
    
    best = study.best_params
    dataset.append({
        'velocity': v_target,
        'altitude': alt_target,
        'best_m': best['m'],
        'best_p': best['p'],
        'best_t': int(best['t']),
        'finesse_max': study.best_value,
        'naca_complet': f"{best['m']}{best['p']}{int(best['t']):02d}"
    })
    
    print(f"✅ Point {i+1}/{N_POINTS_DATASET} | V={v_target:.1f}m/s | Alt={alt_target:.0f}m -> NACA {dataset[-1]['naca_complet']}")

df_airfoil_ai = pd.DataFrame(dataset)
print("\nDataset ready")
display(df_airfoil_ai.head())

✅ Point 1/100 | V=110.8m/s | Alt=11894m -> NACA 9602
✅ Point 2/100 | V=69.1m/s | Alt=5172m -> NACA 8705
✅ Point 3/100 | V=178.4m/s | Alt=6232m -> NACA 9603
✅ Point 4/100 | V=52.6m/s | Alt=10820m -> NACA 9602
✅ Point 5/100 | V=99.0m/s | Alt=9472m -> NACA 8705
✅ Point 6/100 | V=168.1m/s | Alt=2530m -> NACA 9605
✅ Point 7/100 | V=191.7m/s | Alt=10699m -> NACA 7603
✅ Point 8/100 | V=184.8m/s | Alt=5940m -> NACA 8603
✅ Point 9/100 | V=74.9m/s | Alt=1411m -> NACA 9604
✅ Point 10/100 | V=97.6m/s | Alt=5459m -> NACA 8706
✅ Point 11/100 | V=167.6m/s | Alt=8302m -> NACA 9603
✅ Point 12/100 | V=205.8m/s | Alt=3033m -> NACA 8604
✅ Point 13/100 | V=121.8m/s | Alt=7627m -> NACA 9604
✅ Point 14/100 | V=157.3m/s | Alt=2525m -> NACA 9605
✅ Point 15/100 | V=119.3m/s | Alt=7866m -> NACA 9706
✅ Point 16/100 | V=57.7m/s | Alt=7223m -> NACA 8704
✅ Point 17/100 | V=84.2m/s | Alt=5568m -> NACA 8705
✅ Point 18/100 | V=120.0m/s | Alt=5639m -> NACA 9604
✅ Point 19/100 | V=125.5m/s | Alt=7940m -> NACA 9603
✅ Poin

,velocity,altitude,best_m,best_p,best_t,finesse_max,naca_complet
0,110.807430,11894.412737,9,6,2,293.261151,9602
1,69.088232,5171.570982,8,7,5,313.854144,8705
2,178.400418,6232.219322,9,6,3,411.226367,9603
3,52.585610,10820.046290,9,6,2,237.336644,9602
4,99.023021,9472.380854,8,7,5,322.095052,8705


In [6]:
df_airfoil_ai.to_csv("../data/airfoil_optimization_results.csv", index=False)